In [ ]:
import os, glob
import json
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from monai.transforms import (
    Compose, EnsureChannelFirstd, ScaleIntensityRanged,
    Resized, ToTensord, RandRotate90d, RandFlipd,
    RandGaussianNoised, RandZoomd
)
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm
from pathlib import Path

from model_architecture import RSNAResNetCBAMClassifier, GradCAM3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif cihaz: {device}")

# ── Yollar ──────────────────────────────────────────────────────────
BASE          = Path("/Users/ramazanpolat/Desktop/datasets/abdomen")
XLSX          = BASE / "Bilgi.xlsx"
TRAIN_DIR     = BASE / "Egitim Verisi"
WEIGHTS_PATH  = BASE / "rsna2023_best_attention_model.pth"

# ── Sınıf Haritası (config.py ile uyumlu) ───────────────────────────
RAW_TO_SUPER = {
    "Compatible with acute cholecystitis": 0,
    "Gallbladder stone":                   0,
    "Kidney stone":                        1,
    "ureteral stone":                      1,
    "Compatible with acute pancreatitis":  2,
    "Abdominal aortic aneurysm":            3,
    "Abdominal aortic dissection":          3,
    "Compatible with acute appendicitis":  4,
    "Compatible with acute diverticulitis":5,
    "Calcified diverticulum":              5,
}
SUPER_CLASSES = [
    "acute_cholecystitis",
    "kidney_ureter_stone",
    "acute_pancreatitis",
    "aortic_aneurysm_dissection",
    "acute_appendicitis",
    "acute_diverticulitis",
]
TR_NUM_CLASSES = 6  # config.py ile uyumlu


In [ ]:
def build_patient_df(xlsx_path: Path) -> pd.DataFrame:
    """Bilgi.xlsx -> hasta bazlı multi-hot etiket DataFrame."""
    df = pd.read_excel(xlsx_path)
    bb = df[df["Type"] == "Bounding Box"].copy()

    # Her hastanın hangi süper-sınıflara ait olduğunu çıkar
    cases = bb["Case Number"].unique()
    records = []
    for case in cases:
        raw_classes = bb[bb["Case Number"] == case]["Class"].unique()
        label = [0] * TR_NUM_CLASSES
        for rc in raw_classes:
            sid = RAW_TO_SUPER.get(rc)
            if sid is not None:
                label[sid] = 1
        records.append({"case_number": case, **{SUPER_CLASSES[i]: label[i] for i in range(TR_NUM_CLASSES)}})

    patient_df = pd.DataFrame(records)
    print(f"Toplam hasta: {len(patient_df)}")
    print("Sınıf dağılımı:")
    print(patient_df[SUPER_CLASSES].sum().sort_values(ascending=False))
    return patient_df

patient_df = build_patient_df(XLSX)
patient_df.head()


In [ ]:
class TRABDOMENDataset(Dataset):
    """TR_ABDOMEN DICOM hacim dataseti. Flat dizin: TRAIN_DIR/case_number/*.dcm"""
    def __init__(self, df: pd.DataFrame, base_dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _load_volume(self, case_number: int) -> np.ndarray:
        case_dir = self.base_dir / str(case_number)
        dcm_files = sorted(case_dir.glob("*.dcm"))
        if not dcm_files:
            return np.zeros((128, 128, 128), dtype=np.float32)

        slices = [pydicom.dcmread(str(f)) for f in dcm_files]
        slices.sort(key=lambda s: float(getattr(s, "InstanceNumber", 0)))

        volume = np.stack([s.pixel_array for s in slices], axis=-1).astype(np.float32)
        for i, s in enumerate(slices):
            slope     = float(getattr(s, "RescaleSlope",     1.0))
            intercept = float(getattr(s, "RescaleIntercept", 0.0))
            volume[..., i] = volume[..., i] * slope + intercept
        return volume

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        volume = self._load_volume(int(row["case_number"]))
        data_dict = {"image": volume}
        if self.transform:
            data_dict = self.transform(data_dict)
        labels = row[SUPER_CLASSES].values.astype(np.float32)
        return data_dict["image"], torch.tensor(labels)


In [ ]:
train_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250,
                         b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image"], spatial_size=(128, 128, 128)),
    RandRotate90d(keys=["image"], prob=0.5, spatial_axes=(0, 1)),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    RandZoomd(keys=["image"], prob=0.3, min_zoom=0.9, max_zoom=1.1, mode="trilinear"),
    RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.05),
    ToTensord(keys=["image"]),
])

val_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250,
                         b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image"], spatial_size=(128, 128, 128)),
    ToTensord(keys=["image"]),
])

# %80 Train / %20 Val — hasta sızıntısı yok
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(patient_df, groups=patient_df["case_number"]))
df_tr = patient_df.iloc[train_idx]
df_va = patient_df.iloc[val_idx]
print(f"Train: {len(df_tr)} hasta | Val: {len(df_va)} hasta")

train_ds = TRABDOMENDataset(df_tr, TRAIN_DIR, transform=train_transforms)
val_ds   = TRABDOMENDataset(df_va, TRAIN_DIR, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                          num_workers=2, pin_memory=(device.type=="cuda"))
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False,
                          num_workers=2, pin_memory=(device.type=="cuda"))
print("DataLoader'lar hazır.")


In [ ]:
# 1. Boş RSNA modeli (13 sınıf)
model = RSNAResNetCBAMClassifier(num_classes=13)

# 2. Kaggle ağırlıklarını yükle
state = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(state)
print("RSNA ön-eğitim ağırlıkları yüklendi.")

# 3. Backbone'u dondur (özellik çıkarıcı sabit kalır)
for param in model.backbone.parameters():
    param.requires_grad = False

# 4. CBAM dikkat katmanları + classifier eğitilebilir kalır
#    (channel_attention, spatial_attention backbone dışında tanımlı)

# 5. Son katmanı 6 sınıfa göre güncelle
model.custom_classifier = nn.Linear(512, TR_NUM_CLASSES)

model = model.to(device)

# Kaç parametre eğitilebilir?
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Eğitilebilir parametre: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")


In [ ]:
# Sınıf dengesizliği için pos_weight hesapla
counts = patient_df[SUPER_CLASSES].sum().values
neg    = len(patient_df) - counts
pos_weight = torch.tensor(neg / (counts + 1e-8), dtype=torch.float32).to(device)
print("pos_weight:", pos_weight.cpu().numpy().round(2))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Sadece eğitilebilir parametreler için optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-2
)

# Cosine LR Scheduler
num_epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

scaler = GradScaler(device=device.type)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc="Train"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val"):
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


In [ ]:
CKPT_PATH = BASE / "tr_abdomen_finetuned.pth"
best_val_loss = float("inf")
history = {"train": [], "val": []}

print("--- TR_ABDOMEN Fine-Tuning Başlıyor ---")
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss   = validate(model, val_loader, criterion, device)
    scheduler.step()

    history["train"].append(train_loss)
    history["val"].append(val_loss)
    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr_now:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  => En iyi model kaydedildi ({CKPT_PATH.name})")

print(f"En iyi val loss: {best_val_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history["train"], label="Train Loss")
plt.plot(history["val"],   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("TR_ABDOMEN Fine-Tuning Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# En iyi ağırlıkları yükle
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

# register_full_backward_hook (deprecated backward_hook yerine)
target_layer = model.backbone.layer4
gcam = GradCAM3D(model, target_layer)

images, labels = next(iter(val_loader))
input_tensor = images[0:1].to(device)
true_label   = labels[0].numpy()

# Modelin en yüksek skorlu sınıfını hedef al
with torch.no_grad():
    logits = model(input_tensor)
predicted_class = logits[0].argmax().item()
print(f"Gerçek: {[SUPER_CLASSES[i] for i,v in enumerate(true_label) if v==1]}")
print(f"Tahmin sınıfı: {SUPER_CLASSES[predicted_class]}")

heatmap_3d = gcam.generate_heatmap(input_tensor, class_idx=predicted_class)

slice_idx = heatmap_3d.shape[2] // 2  # Hacmin ortası
raw_slice = input_tensor[0, 0, :, :, slice_idx].cpu().numpy()
cam_slice = heatmap_3d[:, :, slice_idx]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(raw_slice, cmap="bone");    axes[0].set_title("CT Kesit");        axes[0].axis("off")
axes[1].imshow(cam_slice, cmap="jet");     axes[1].set_title("Grad-CAM");        axes[1].axis("off")
axes[2].imshow(raw_slice, cmap="bone");    axes[2].set_title("Overlay")
axes[2].imshow(cam_slice, cmap="jet", alpha=0.5); axes[2].axis("off")
plt.suptitle(f"Tahmin: {SUPER_CLASSES[predicted_class]}")
plt.tight_layout()
plt.show()
